In [ ]:
!pip install spotipy --quiet

import os
import time
import pandas as pd
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.2/354.2 kB 9.1 MB/s eta 0:00:00


In [ ]:
CLIENT_ID = "YOUR_SPOTIFY_CLIENT_ID"
CLIENT_SECRET = "YOUR_SPOTIFY_CLIENT_SECRET"

auth_manager = SpotifyClientCredentials(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET
)
sp = spotipy.Spotify(auth_manager=auth_manager)


In [ ]:
import pandas as pd

df = pd.read_csv("music_data.csv")
df.head()
df.columns

Index(['track_id', 'name', 'artist', 'spotify_preview_url', 'spotify_id',
       'tags', 'genre', 'year', 'duration_ms', 'danceability', 'energy', 'key',
       'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
       'liveness', 'valence', 'tempo', 'time_signature'],
      dtype='object')

In [ ]:
spotify_ids = (
    df['spotify_id']
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

len(spotify_ids)

50674

In [ ]:
import time
import spotipy

def fetch_tracks_popularity(id_batch):
    """
    Given a list of up to 50 spotify track IDs, return
    a DataFrame with spotify_id and popularity (0–100).
    """
    results = sp.tracks(id_batch)
    rows = []
    for t in results['tracks']:
        if t is None:
            continue
        rows.append({
            "spotify_id": t['id'],
            "popularity": t['popularity'],
            "spotify_name_api": t['name'],
            "spotify_artist_api": ", ".join([a['name'] for a in t['artists']])
        })
    return pd.DataFrame(rows)

all_pop_frames = []
batch_size = 50

for i in range(0, len(spotify_ids), batch_size):
    batch = spotify_ids[i:i+batch_size]
    pop_df_batch = fetch_tracks_popularity(batch)
    all_pop_frames.append(pop_df_batch)
    time.sleep(0.2)   # be polite to the API

popularity_df = pd.concat(all_pop_frames, ignore_index=True)
popularity_df.head()

,spotify_id,popularity,spotify_name_api,spotify_artist_api
0,09ZQ5TmUG8TSL56n0knqrj,0,Mr. Brightside,The Killers
1,06UfBBDISthj1ZJAtX4xjj,0,Wonderwall,Oasis
2,0keNu0t0tqsWtExGM3nT1D,1,Come As You Are,Nirvana
3,0ancVQ9wEcHVd0RrGICTE4,2,Take Me Out,Franz Ferdinand
4,01QoK9DA7VTeTSE3MNzp4I,0,Creep,Radiohead


In [ ]:
df = df.merge(popularity_df, on="spotify_id", how="left")
df[['spotify_id', 'popularity']].head()
df['popularity'].describe()

,popularity
count,50683.000000
mean,7.072648
std,14.301065
min,0.000000
25%,0.000000
50%,0.000000
75%,5.000000
max,89.000000


In [ ]:
df['hit_label'] = (df['popularity'] >= 60).astype(int)
df['hit_label'].value_counts()

,count
hit_label,
0,50202
1,481


In [ ]:
def popularity_bucket(p):
    if pd.isna(p):
        return "unknown"
    elif p < 40:
        return "emerging"
    elif p < 70:
        return "moderately_popular"
    else:
        return "viral"

df['popularity_level'] = df['popularity'].apply(popularity_bucket)
df['popularity_level'].value_counts()

,count
popularity_level,
emerging,47885
moderately_popular,2654
viral,144


In [ ]:
df[['spotify_id', 'popularity', 'hit_label', 'popularity_level']].head()

,spotify_id,popularity,hit_label,popularity_level
0,09ZQ5TmUG8TSL56n0knqrj,0,0,emerging
1,06UfBBDISthj1ZJAtX4xjj,0,0,emerging
2,0keNu0t0tqsWtExGM3nT1D,1,0,emerging
3,0ancVQ9wEcHVd0RrGICTE4,2,0,emerging
4,01QoK9DA7VTeTSE3MNzp4I,0,0,emerging


In [ ]:
# Create a dictionary that maps each column to a feature type
feature_group_map = {}

# Audio
for col in audio_features:
    feature_group_map[col] = 'Audio'

# Metadata
for col in metadata_features:
    feature_group_map[col] = 'Metadata'

# Social Engagement features
for col in social_features:
    feature_group_map[col] = 'Social_Engagement'

# Add target columns
feature_group_map['hit_label'] = 'Target'
feature_group_map['popularity_level'] = 'Target'
feature_group_map['popularity'] = 'Target'

In [ ]:
df_long = df.melt(id_vars=['spotify_id'],
                  value_vars=list(feature_group_map.keys()),
                  var_name='feature_name',
                  value_name='feature_value')

df_long['feature_type'] = df_long['feature_name'].map(feature_group_map)

In [ ]:
df_attributes = pd.DataFrame(list(feature_group_map.items()), columns=['feature_name', 'feature_type'])

In [ ]:
df_final = df.copy()
df_final['popularity'] = df['popularity']
df_final['hit_label'] = df['hit_label']
df_final['popularity_level'] = df['popularity_level']

In [ ]:
df_final.head()
df_final.columns

Index(['track_id', 'name', 'artist', 'spotify_preview_url', 'spotify_id',
       'tags', 'genre', 'year', 'duration_ms', 'danceability', 'energy', 'key',
       'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
       'liveness', 'valence', 'tempo', 'time_signature', 'popularity',
       'spotify_name_api', 'spotify_artist_api', 'hit_label',
       'popularity_level'],
      dtype='object')

In [ ]:
df_final.to_csv("final_music_dataset_with_popularity_and_labels.csv", index=False)

In [ ]:
# Save the enriched dataset to CSV
df_final.to_csv('final_music_dataset.csv', index=False)

# For Kaggle/Colab: Generate download link
from google.colab import files
files.download('final_music_dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>